# viel Face Cloak service
Use this notebook as the backend for your website. **All photos and proof ZIPs are selected on the website. There are no notebook photo-upload cells.**

This integrates upstream Fawkes for facial-recognition training privacy. It does not protect against Meta AI editing. It uses an isolated Python 3.8 / TensorFlow 2.4 environment on CPU, so a GPU is not required for cloaking. Installation and processing can take several minutes. The independent recognition evaluator uses the host's PyTorch.

Read the upstream documentation and licence: https://github.com/Shawn-Shan/fawkes . This is a research integration, without SANDLab endorsement. Compatibility and model downloads are checked during setup.


In [ ]:
import pathlib, os, subprocess, sys
repo=pathlib.Path('/content/viel-face-service')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/indujavg/viel.git',str(repo)],check=True)
os.chdir(repo)
if str(repo) not in sys.path: sys.path.insert(0,str(repo))
subprocess.run([sys.executable,'scripts/setup_face_colab.py'],check=True)
os.environ['VIEL_FAWKES_PYTHON']='/content/viel-fawkes-py38/bin/python'
os.environ['VIEL_FAWKES_SOURCE']='/content/fawkes'
print('Setup finished. Run the service cell next.')


In [ ]:
import secrets, threading, time, re
import requests, uvicorn
from backend.face_app import create_face_app
from backend.faces.fawkes import FawkesRunner
from google.colab import files
assert 'viel_face_server' not in globals(), 'Service already started. Use its existing configuration or restart the runtime before rerunning.'
token=secrets.token_urlsafe(32)
runner=FawkesRunner()
viel_face_server=uvicorn.Server(uvicorn.Config(create_face_app(runner=runner,token=token),host='127.0.0.1',port=8000))
threading.Thread(target=viel_face_server.run,daemon=True).start()
for _ in range(30):
    try:
        response=requests.get('http://127.0.0.1:8000/face/health',headers={'Authorization':f'Bearer {token}'},timeout=2)
        if response.ok and response.json().get('ready'): break
    except requests.RequestException: pass
    time.sleep(1)
else: raise RuntimeError('Fawkes service did not start. Inspect the output above.')
binary=pathlib.Path('/content/viel-cloudflared')
response=requests.get('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',timeout=60)
response.raise_for_status();binary.write_bytes(response.content);binary.chmod(0o755)
log=pathlib.Path('/content/viel-face-tunnel.log');log_file=log.open('w')
viel_face_tunnel=subprocess.Popen([str(binary),'tunnel','--url','http://127.0.0.1:8000'],stdout=log_file,stderr=subprocess.STDOUT)
for _ in range(60):
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log.read_text())
    if match: break
    time.sleep(1)
else: raise RuntimeError('Tunnel failed. Inspect viel-face-tunnel.log.')
config=pathlib.Path('/content/viel-backend.env')
config.write_text(f'VIEL_BACKEND_URL={match.group(0)}\nVIEL_API_TOKEN={token}\n')
files.download(str(config))
print('Service started. Keep this runtime connected. Select images only on your website.')


## Connect on Windows
In the website terminal, press Ctrl+C, then run:
```powershell
cd "$env:USERPROFILE\viel"
git pull --ff-only
npm.cmd ci
Move-Item "$env:USERPROFILE\Downloads\viel-backend.env" ".env.local" -Force
npm.cmd run dev -- --host 127.0.0.1
```
Open the printed Local URL, then click **Check Fawkes connection**. The downloaded file contains a private token: do not share it.

Choose **Cloak a face photo** to generate an image. Choose **Run a recognition experiment** to upload a labelled train/test ZIP from the website. See `docs/face-proof.md` for the exact layout and interpretation. A cloak download alone is not evidence of protection.
